# Analizar un partido con GPU en Colab — tracking JSON

Genera el **tracking JSON** de un partido (jugadores + balón + posesión) usando la GPU gratis de Colab. Es el mismo análisis que `video_track.py`, pero con GPU: 10-20x más rápido que en CPU.

**Antes de ejecutar:** pon el entorno en GPU — menú **Entorno > Cambiar tipo de entorno > GPU (T4)**. Sin GPU el análisis también funciona (device="cpu") pero tarda horas.

**El vídeo debe estar en Google Drive** (súbelo antes si no está). El tracking JSON resultante queda en Drive y se usa en la pantalla de análisis manual (la web `analizador_video.html` o la pantalla de Vídeo de la app).

**Qué hace el análisis:** solo tracking + estadísticas (posesión, territorio, balón). NO clasifica eventos (tiros, goles, pases, faltas): las jugadas las marcas tú a mano en la pantalla manual.

In [ ]:
# 1. Instalar ultralytics (YOLO) y comprobar la GPU
!pip install -q ultralytics
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('SIN GPU: cambia el entorno a GPU (Entorno > Cambiar tipo de entorno > GPU).')

In [ ]:
# 2. Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Bajar el código actualizado del repo
!git clone -q https://github.com/Josen-IG/Match_view.git /content/repo
import sys
sys.path.insert(0, '/content/repo')
import video_track as vt
print('video_track cargado desde /content/repo')

In [ ]:
# 4. CONFIGURACIÓN — edita estos valores para tu partido
VIDEO_DRIVE = '/content/drive/MyDrive/Git Hub/video.mp4'  # ruta del vídeo en Drive
TEAM_LOCAL = 'C.A. Cirbonero "B"'
TEAM_VISITANTE = 'C.A. Milagres de Futbol'
COLOR_LOCAL = '#1E88E5'        # Cirbonero: azul
COLOR_VISITANTE = '#E53935'    # Milagres: rojo
COLOR_GK_LOCAL = '#000000'     # portero local: negro
COLOR_GK_VISITANTE = '#1E8E3E' # portero visitante: verde
COLOR_ARBITRO = '#CCFF00'      # árbitro: amarillo fosfo
KICKOFF_S = 1026.0             # saque inicial en segundos de vídeo (17:06)
ALCANCE_MIN = None             # None = partido completo; 10 = primeros 10 min (prueba)
YOLO_MODEL = 'yolo11s.pt'      # yolo11n más rápido, yolo11m más preciso
SAMPLE_FPS = 5                 # 5 fps para partido completo (10 para prueba corta)
FORMATION_LOCAL = '1-4-4-2'
FORMATION_VISITANTE = '1-4-3-3'

In [ ]:
# 5. Analizar con GPU (device='cuda')
import torch
from pathlib import Path
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE)
video = Path(VIDEO_DRIVE)
assert video.exists(), f'No existe el vídeo: {VIDEO_DRIVE}'
informe = vt.analyze_video_full(
    video,
    color_local=COLOR_LOCAL, color_visitante=COLOR_VISITANTE,
    color_gk_local=COLOR_GK_LOCAL, color_gk_visitante=COLOR_GK_VISITANTE,
    color_arbitro=COLOR_ARBITRO,
    team_local=TEAM_LOCAL, team_visitante=TEAM_VISITANTE,
    sample_fps=SAMPLE_FPS, yolo_model=YOLO_MODEL,
    max_minutes=ALCANCE_MIN, kickoff_offset=KICKOFF_S,
    camera_mode='seguimiento', device=DEVICE,
    formation_local=FORMATION_LOCAL, formation_visitante=FORMATION_VISITANTE,
)
print('Muestras:', informe['samples_analyzed'])
print('Posesión:', informe['estadisticas']['posesion'])
print('Tracking JSON:', informe.get('tracking_json'))

In [ ]:
# 6. Copiar el tracking JSON a la carpeta "Git Hub" de Drive
import shutil, json
from pathlib import Path
tracking = Path(informe['tracking_json'])
OUT = Path('/content/drive/MyDrive/Git Hub')
OUT.mkdir(parents=True, exist_ok=True)
destino = OUT / f'{tracking.stem}.json'
shutil.copy(tracking, destino)
print('Tracking JSON en Drive:', destino)
print('Tamaño (MB):', round(destino.stat().st_size / 1e6, 1))
tj = json.loads(destino.read_text())
print('Muestras:', len(tj['samples']), '| eventos:', len(tj['events']))

## Usar el tracking JSON en la pantalla manual

1. Descarga el tracking JSON de Drive (o déjalo en la carpeta "Git Hub").
2. En la app (pantalla de Vídeo) o en `analizador_video.html`, carga el vídeo y el tracking JSON: verás las cajas, el balón y la trayectoria encima del vídeo, la posesión en vivo y las herramientas para señalar jugadas (flechas, círculos, texto) y marcar cortes.
3. Todo lo que anotes se exporta en JSON al final.

**Notas:**
- Si Colab se desconecta a mitad (límite de sesión gratuita), vuelve a ejecutar: el análisis es por tramos y se puede reanudar, o usa la VM con `chunked_full.py` (más lento pero resistente a reinicios).
- El modelo de keypoints del campo (`pitch_keypoints.pt`) se descarga solo la primera vez de HuggingFace (~86 MB); sin él el análisis sigue igual, solo sin coordenadas de campo.